# Notebook 07: Deep Learning - LSTM Forecasting

## Project: Indian Air Quality Index (AQI) Comprehensive Analysis
## BTech Final Year Project - Data Science & Machine Learning (8th Semester)

### Objective:
Build Long Short-Term Memory (LSTM) neural network for AQI time series forecasting. LSTM captures long-term temporal dependencies in sequential data.

### Prerequisites:
- Complete Notebook 01 (generates city_day_cleaned.csv)
- Libraries: pandas, numpy, tensorflow/keras, scikit-learn

### Run Time: 30-45 minutes (GPU recommended)

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported!')

### Explanation:

- **from tensorflow.keras.models import Sequential**: Linear stack of neural network layers.

- **from tensorflow.keras.layers import LSTM**: Long Short-Term Memory layer (handles sequential data, remembers long-term patterns).

- **from tensorflow.keras.layers import Dense**: Fully connected layer (standard neural network layer).

- **from tensorflow.keras.layers import Dropout**: Regularization layer (randomly drops neurons to prevent overfitting).

- **from tensorflow.keras.callbacks import EarlyStopping**: Stops training when validation loss stops improving (prevents overfitting).

In [ ]:
data_path = os.path.join('..', 'datasets', 'city_day_cleaned.csv')
df = pd.read_csv(data_path, parse_dates=['Datetime'])
df = df.sort_values('Datetime').reset_index(drop=True)
print(f'Loaded {len(df)} records')

## Step 2: Prepare Time Series Data

In [ ]:
daily_aqi = df.groupby('Datetime')['AQI'].mean().resample('D').mean()
daily_aqi = daily_aqi.interpolate(method='linear')
data = daily_aqi.values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)
print(f'Time series length: {len(scaled_data)} days')
print(f'Scaled range: [{scaled_data.min():.2f}, {scaled_data.max():.2f}]')

### Explanation:

- **MinMaxScaler(feature_range=(0, 1))**: Scales data to range [0, 1] (neural networks work better with normalized data).

- **fit_transform()**: Learns min/max values AND applies scaling.

## Step 3: Create Sequences for LSTM

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)
seq_length = 30
X, y = create_sequences(scaled_data, seq_length)
print(f'Sequences created: {X.shape[0]}')
print(f'Sequence shape: {X.shape} (samples, time steps, features)')
print(f'Target shape: {y.shape}')

### Explanation:

- **seq_length=30**: Uses past 30 days to predict next day.

- **X.append(data[i:i+seq_length])**: Creates sliding window of 30 days.

- **y.append(data[i+seq_length])**: Target is the value after the 30-day window.

- **Shape (samples, timesteps, features)**: LSTM expects 3D input.

## Step 4: Train/Test Split

In [ ]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
print(f'Training sequences: {len(X_train)}')
print(f'Test sequences: {len(X_test)}')

## Step 5: Build LSTM Model

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.summary()

### Explanation:

- **LSTM(64, return_sequences=True)**: First LSTM layer with 64 units. return_sequences=True means output full sequence (for stacking layers).

- **input_shape=(seq_length, 1)**: Input shape is (30 timesteps, 1 feature).

- **Dropout(0.2)**: Drops 20% of neurons randomly during training (prevents overfitting).

- **LSTM(32, return_sequences=False)**: Second LSTM layer with 32 units. Returns only last output.

- **Dense(16, activation='relu')**: Fully connected layer with ReLU activation.

- **Dense(1)**: Output layer with 1 neuron (predicts next day AQI).

- **optimizer='adam'**: Adaptive Moment Estimation optimizer (efficient gradient descent).

- **loss='mse'**: Mean Squared Error loss function (for regression).

## Step 6: Train LSTM Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_split=0.2, epochs=100, 
                    batch_size=32, callbacks=[early_stop], verbose=1)
print('Training completed!')

### Explanation:

- **EarlyStopping(monitor='val_loss', patience=10)**: Stops training if validation loss doesn't improve for 10 epochs.

- **restore_best_weights=True**: Loads best model weights after early stopping.

- **validation_split=0.2**: Uses 20% of training data for validation.

- **epochs=100**: Maximum 100 training iterations (early stopping may end earlier).

- **batch_size=32**: Processes 32 samples before updating weights.

## Step 7: Plot Training History

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch', fontsize=12, fontweight='bold')
plt.ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
plt.title('LSTM Training History', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8: Make Predictions & Evaluate

In [ ]:
y_pred_scaled = model.predict(X_test)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
mae = mean_absolute_error(y_test_actual, y_pred)
r2 = r2_score(y_test_actual, y_pred)
print(f'LSTM Model Performance:')
print(f'RMSE: {rmse:.2f}')
print(f'MAE: {mae:.2f}')
print(f'R²: {r2:.4f}')

### Explanation:

- **model.predict()**: Generates predictions on test data.

- **inverse_transform()**: Converts scaled predictions back to original AQI scale.

## Step 9: Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(range(len(y_test_actual)), y_test_actual, label='Actual AQI', linewidth=2)
plt.plot(range(len(y_pred)), y_pred, label='Predicted AQI', linewidth=2, alpha=0.7)
plt.xlabel('Days', fontsize=12, fontweight='bold')
plt.ylabel('AQI', fontsize=12, fontweight='bold')
plt.title('LSTM AQI Forecasting - Actual vs Predicted', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10: Future Forecasting (Next 30 Days)

In [ ]:
last_sequence = scaled_data[-seq_length:]
forecast_days = 30
forecast = []
for i in range(forecast_days):
    X_forecast = last_sequence.reshape(1, seq_length, 1)
    next_pred = model.predict(X_forecast)
    forecast.append(next_pred[0, 0])
    last_sequence = np.append(last_sequence[1:], next_pred)
forecast_actual = scaler.inverse_transform(np.array(forecast).reshape(-1, 1))
print(f'30-day forecast generated!')
print(f'Forecast range: [{forecast_actual.min():.0f}, {forecast_actual.max():.0f}]')

### Explanation:

- **last_sequence**: Uses last 30 days as starting sequence.

- **Iterative prediction**: Each prediction becomes input for next prediction.

- **forecast_days=30**: Predicts next 30 days.

## Step 11: Save LSTM Model & Results

In [ ]:
model.save(os.path.join('..', 'models', 'lstm_model.h5'))
forecast_df = pd.DataFrame({'Day': range(1, forecast_days+1), 'Forecasted_AQI': forecast_actual.flatten()})
forecast_df.to_csv(os.path.join('..', 'outputs', 'lstm_forecast.csv'), index=False)
print('LSTM model and forecast saved!')
print('READY FOR NOTEBOOK 08 (SHAP Analysis)')

## Summary

Built LSTM neural network:
1. Data preparation and scaling
2. Sequence creation (30-day windows)
3. Model architecture (2 LSTM layers + Dense layers)
4. Training with early stopping
5. Evaluation (RMSE, MAE, R²)
6. 30-day future forecasting

**Architecture:**
- LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(16) → Dense(1)

**Next**: Notebook 08 - SHAP Model Interpretation